# Optional: ARIMA and LSTM comparison (aggregate demand)

The Tredence brief asks for ARIMA and LSTM models. Fitting one ARIMA/LSTM per store-item (500 series) is slow and unnecessary here because the series share the same seasonal shape, so this notebook compares the classical and deep-learning models on **total daily demand** over the same unseen quarter (2017-10-01 to 2017-12-29, trained on data up to 2017-09-30) as the gradient-boosting model in notebook 01.

**Requirements:** `pip install statsmodels tensorflow` (run locally or in Google Colab; they are not needed for the app).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from forecasting import smape, metrics

df = pd.read_csv(ROOT / "data" / "train.csv", parse_dates=["date"])
total = df.groupby("date")["sales"].sum().asfreq("D")
cut = pd.Timestamp("2017-09-30")
train = total[:cut]
test = total["2017-10-01":"2017-12-29"]
ref = pd.read_csv(ROOT / "reports" / "fold_b_daily_total.csv", index_col=0, parse_dates=True)   # saved HGB + naive forecasts
print(len(train), "training days,", len(test), "test days")

## ARIMA
- **SARIMA(1,1,1)(1,1,1)7**: weekly seasonality only (period 7). It cannot represent the annual cycle.
- **SARIMAX + annual Fourier terms**: the same model with two sine/cosine pairs for the yearly cycle as external regressors.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def fourier(idx, K=2):
    t = idx.dayofyear.values
    return np.column_stack([f(2 * np.pi * k * t / 365.25) for k in range(1, K + 1) for f in (np.sin, np.cos)])

y = np.log(train.iloc[-730:])                      # last two years, log scale
sarima = SARIMAX(y, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_fc = pd.Series(np.exp(sarima.forecast(len(test)).values), index=test.index)

sarimax = SARIMAX(y, exog=fourier(y.index), order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarimax_fc = pd.Series(np.exp(sarimax.forecast(len(test), exog=fourier(test.index)).values), index=test.index)
print("SARIMA SMAPE %.2f%% | SARIMAX+Fourier SMAPE %.2f%%" % (smape(test, sarima_fc), smape(test, sarimax_fc)))

## LSTM
A direct multi-step LSTM: the input is the previous 90 days (scaled sales + day-of-year sine/cosine), the output is the next 90 days in one shot (same no-recursion idea as the gradient-boosting model).

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

tf.random.set_seed(42); np.random.seed(42)
IN, OUT = 90, 90

def channels(series_idx, scaled):
    t = series_idx.dayofyear.values
    return np.column_stack([scaled, np.sin(2 * np.pi * t / 365.25), np.cos(2 * np.pi * t / 365.25)])

scaler = MinMaxScaler().fit(train.values.reshape(-1, 1))
scaled = scaler.transform(train.values.reshape(-1, 1)).ravel()
feats = channels(train.index, scaled)

X, Yt = [], []
for i in range(IN, len(train) - OUT + 1):
    X.append(feats[i - IN:i]); Yt.append(scaled[i:i + OUT])
X, Yt = np.array(X), np.array(Yt)
split = int(len(X) * 0.9)                          # time-ordered validation for early stopping

model = tf.keras.Sequential([tf.keras.layers.Input((IN, 3)), tf.keras.layers.LSTM(64), tf.keras.layers.Dense(OUT)])
model.compile(optimizer="adam", loss="mse")
model.fit(X[:split], Yt[:split], validation_data=(X[split:], Yt[split:]), epochs=80, batch_size=32, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)])

pred = model.predict(feats[-IN:][None, ...], verbose=0)[0]
lstm_fc = pd.Series(scaler.inverse_transform(pred.reshape(-1, 1)).ravel()[:len(test)], index=test.index)
print("LSTM SMAPE %.2f%%" % smape(test, lstm_fc))

## Comparison (total daily demand, unseen quarter)

In [ ]:
results = pd.DataFrame({
    "Seasonal naive x growth": metrics(test, ref["snaive"].reindex(test.index)),
    "SARIMA (weekly)": metrics(test, sarima_fc),
    "SARIMAX + annual Fourier": metrics(test, sarimax_fc),
    "LSTM (direct 90-day)": metrics(test, lstm_fc),
    "Gradient boosting (notebook 01)": metrics(test, ref["hgb"].reindex(test.index)),
}).T
display(results)
test.plot(label="actual", figsize=(12, 4), color="black")
for name, s in [("SARIMA", sarima_fc), ("SARIMAX+Fourier", sarimax_fc), ("LSTM", lstm_fc), ("Gradient boosting", ref["hgb"])]:
    s.reindex(test.index).plot(label=name, alpha=0.8)
plt.legend(); plt.title("Total daily demand: actual vs models"); plt.show()

Paste the results table into the README. Expect the weekly-only SARIMA to struggle on this quarter because of the annual cycle; the Fourier terms and the gradient-boosting model, which sees last year's values directly, should do much better. Write your own conclusion from the numbers you get.